In [0]:
from pyspark.sql.functions import col, current_timestamp, round, when
from pyspark.sql.utils import AnalysisException

print("Bắt đầu tiến trình Transformation (Bronze -> Silver)...")

try:
    # 1. Đọc dữ liệu từ lớp Bronze (Nguồn)
    bronze_table = "workspace.stock_db.bronze_stock_prices"
    df_bronze = spark.read.table(bronze_table)
    
    # 2. Làm sạch dữ liệu (Data Cleaning)
    df_clean = df_bronze.dropna(subset=["symbol", "date", "close"]) \
                        .dropDuplicates(["symbol", "date"])
    
    # 3. Biến đổi dữ liệu (Data Transformation)
    df_silver = df_clean.withColumn("price_change", round(col("close") - col("open"), 2)) \
                        .withColumn("is_positive", when(col("close") > col("open"), True).otherwise(False)) \
                        .withColumn("processed_at", current_timestamp())
    
    # 4. Chọn các cột cần thiết
    final_columns = [
        "symbol", "date", "open", "high", "low", "close", 
        "volume", "adj_close", "price_change", "is_positive", "processed_at"
    ]
    df_silver = df_silver.select(*final_columns)
    
    # 5. Ghi dữ liệu vào lớp Silver (ĐÚNG TÊN BẢNG THEO KẾ HOẠCH)
    silver_table = "workspace.stock_db.silver_prices"
    
    df_silver.write.format("delta") \
             .mode("overwrite") \
             .option("mergeSchema", "true") \
             .saveAsTable(silver_table)
             
    print(f"--- THÀNH CÔNG: Đã xử lý và lưu dữ liệu vào {silver_table} ---")
    
    # 6. Hiển thị kết quả kiểm tra
    display(spark.sql(f"SELECT symbol, date, close, price_change, is_positive FROM {silver_table} ORDER BY date DESC LIMIT 10"))

except AnalysisException as e:
    print(f"Lỗi: Không tìm thấy bảng Bronze. Chi tiết: {e}")
except Exception as e:
    print(f"Đã xảy ra lỗi: {e}")